<div style="color: hotpink; font-size: 40px; font-family: Times; font-weight: bold;">
Grove IMU Sensor V2</div>

<div style="background-color: #f3e5ff; padding: 12px; border-radius: 10px; font-size: 16px; font-family: Times;">
This notebook will help you understand how a Grove IMU Sensor works and functions.</div>

<div style="background-color: #7ed4e6; padding: 12px; border-radius: 10px; font-size: 16px; font-family: Times;">
<h3 style="font-size: 20px; font-weight: bold;">Grove IMU:</h3>
<ul> <li>IMU: Inertial Measurement Unit</li> </ul>
<ul> <li>the shield converts the PYNQ-Z2’s Arduino-style pins into convenient four-wire Grove sockets, so you can connect sensors without using individual jumper wires</li> </ul>
    
<b>IMU measures:</b>
<ul> <li>acceleration</li> </ul>
<ul> <li>rotation speed</li> </ul>
<ul> <li>magnetic direction</li> </ul>
<l><b>the program below will combine these measurements to calculate:</b></l>
<ul> <li><b>roll:</b> rotation around x-axis</li> </ul>
<ul> <li><b>pitch:</b> rotation around y-axis</li> </ul>
<ul> <li><b>yaw:</b> rotation around z-axis</li> </ul>
    
<ul> <li>those three angles are used to rotate a set of X, Y, and Z arrows in a 3D graph</li> </ul>

</div>

<div style="background-color: #71cae0; padding: 12px; border-radius: 10px; font-size: 16px; font-family: Times;">
<h3 style="font-size: 20px; font-weight: bold;">Import Libraries:</h3>

<ul> <li><b>time:</b> measure how long each loop takes/pauses program between sensor readings</li> </ul> 
<ul> <li><b>math:</b> provides mathematical functions to calculate angles & rotation matrices</li> </ul>
<ul> <li><b>numpy:</b> creates vectors, matrices, matric multiplication operations</li> </ul>
<ul> <li><b>matplotlib.pyplot:</b> creates & updates 3D graph</li> </ul>

<br>
<ul> <li><b>mpl_toolkits.mplot3d:</b> provides Matplotlib's 3D plotting support</li> </ul>
<ul> <li><b>IPython.display:</b> updates the graph inside Jupyter Notebook</li> </ul>
<ul> <li><b>pynq.overlays.base:</b> imports the standard PYNQ-Z2 base overlay</li> </ul>
<ul> <li><b>pynq_peripherals:</b> imports the adapter used to communicate with a Grove sensor connected through the PYNQ-Z2 Arduino header</li> </ul>


    
</div>

In [ ]:
import time
import math
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import display, clear_output
from pynq.overlays.base import BaseOverlay
from pynq_peripherals import ArduinoSEEEDGroveAdapter

<div style="background-color: #63bfd9; padding: 12px; border-radius: 10px; font-size: 16px; font-family: Times;">
<h3 style="font-size: 20px; font-weight: bold;">Initialize:</h3>

<ul> <li><b>base:</b> creates an adapter for a Grove IMU connected to the Arduino connector, tells the library that the sensor is connected through the Arduino header, & tells the adapter that the connected I2C device is a Grove IMU</li> </ul> 
<ul> <li><b>adapter:</b> provides mathematical functions to calculate angles & rotation matrices</li> </ul>
<ul> <li><b>imu:</b> stores the IMU object in the variable imu</li> </ul>


</div>

In [ ]:
base = BaseOverlay('base.bit')
adapter = ArduinoSEEEDGroveAdapter(base.ARDUINO, I2C='grove_imu')
imu = adapter.I2C

<div style="background-color: #71cae0; padding: 12px; border-radius: 10px; font-size: 16px; font-family: Times;">
<h3 style="font-size: 20px; font-weight: bold;">Set Filter Parameters, Take Accelerometer Reading, & Initialize Figure:</h3>

<ul> <li><b>alpha:</b> controls the complementary filter, 98% of the result comes from the gyroscope, 2% comes from accelerator</li> </ul>
<ul><ul><ul> <li>gyroscope responds quickly but slowly drifts over time, the accelerometer does not drift as much, but its readings can become noisy when the sensor moves, so combining them gives a more stable result</li></ul></ul></ul>
<ul> <li><b>dt:</b> the desired time between sensor readings, the program attempts to update approximately 20 times per second</li> </ul>
<ul> <li><b>roll_filtered/pitch_filtered:</b> stores the current filtered roll and pitch angles, starting at 0 and updating during every loop</li> </ul>

<br>
<ul> <li><b>imu.fetch_motion9():</b> tells the IMU to collect a new set of measurements, motion9 referring to nine measurements</li> </ul>
<ul> <li><b>ax/ay/az:</b> stores the three acceleration readings, when the sensor is not moving, the accelerometer mainly detects gravity</li> </ul>
<ul> <li><b>if az == 0:</b>prevents mathematical problems if az is exactly zero</li> </ul>
<ul> <li><b>roll_filtered/pitch_filtered:</b> intial roll & pitch calculation</li> </ul>
<ul> <li><b>57.29578:</b> converts radians to degrees</li> </ul>

<br><ul> <li><b>plt.ion():</b> enables Matplotlib’s interactive mode, allowing the graph to be updated repeatedly while the program runs</li> </ul>
<ul> <li><b>fig:</b> creates a figure that is seven inches wide and seven inches tall</li> </ul>
<ul> <li><b>origin:</b> represents the center of the graph</li> </ul>
<ul> <li><b>x_axis_base/y_axis_base/z_axis_base:</b> unit vectors beginning in their normal positions, the rotation matrix later changes their directions</li> </ul>
    
</div>

In [ ]:
# 2. initialize the filter settings and angle variables
alpha = 0.98       # complementary filter coefficient
dt = 0.05          # sampling interval in seconds

roll_filtered = 0.0
pitch_filtered = 0.0


# 3. take the first reading to initialize the angles
imu.fetch_motion9()

ax, ay, az = imu.accel_x, imu.accel_y, imu.accel_z

# prevent division or calculation problems when az is zero
if az == 0:
    az = 0.000001

roll_filtered = math.atan2(ay, az) * 57.29578

pitch_filtered = math.atan2(
    -ax,
    math.sqrt(ay * ay + az * az)
) * 57.29578


# 4. initialize the Matplotlib 3D figure
plt.ion()

fig = plt.figure(figsize=(7, 7))


# define the original X, Y, and Z unit vectors
origin = np.array([0, 0, 0])

x_axis_base = np.array([1, 0, 0])
y_axis_base = np.array([0, 1, 0])
z_axis_base = np.array([0, 0, 1])


print(
    "Starting the 3D attitude simulation...\n"
    "Try rotating the sensor!\n"
    "Press the Jupyter Stop button or Ctrl+C to stop.\n"
)

<div style="background-color: #71cae0; padding: 12px; border-radius: 10px; font-size: 16px; font-family: Times;">
<h3 style="font-size: 20px; font-weight: bold;">Building the Interactive Graph:</h3>

<ul> <li><b>start_time:</b> records the starting time of the current loop, will be used later to calculate how long the calculations and graph update took</li> </ul>
<ul> <li><b>imu.fetch_motion9():</b> requests a fresh measurement from the sensor of the nine values</li> </ul>
<ul> <li><b>ax/ay/az:</b> measures acceleration along three directions, calculating roll & pitch from gravity</li> </ul>
<ul> <li><b>gx/gy/gz:</b> measures angular velocity along three directions, helping to update roll & pitch</li> </ul>
<ul> <li><b>mx/my/mz:</b> measures magnetic field along three directions, acting similar to a digital compass and is used to estimate yaw</li> </ul>


<br>
<ul> <li><b>roll_acc/pitch_acc:</b> convert roll & pitch to radians</li> </ul>
<ul> <li><b>roll_filtered/pitch_filtered:</b>calculates the estimated roll & pitch change since the previous readings</li> </ul>


<br>
<ul> <li><b>r_rad/p_rad:</b> enables Matplotlib’s interactive mode, allowing the graph to be updated repeatedly while the program runs</li> </ul>
<ul> <li><b>Xh/Yh:</b> calculates a corrected horizontal X magnetic component & Y magnetic component</li> </ul>
<ul> <li><b>yaw:</b> calculates yaw from the corrected magnetometer measurements, representing compass-like rotation around the vertical z-axis</li> </ul>
<ul> <li><b>if yaw &lt; 0:</b> ensures yaw stays in range of 0 to 360 degrees</li> </ul>
<ul> <li><b>y_rad; 0:</b> convert to radians, so it can be used in the rotation matrix</li> </ul>

<br>
<ul> <li><b>R_x/R_y/R_z:</b> rotates an object around the x-axis (roll), y-axis (pitch), & z-axis (yaw)</li> </ul>
    <ul><ul><ul> <li>during whichever axis rotation, the corresponding axis remains fixed while the other two axes change</li></ul></ul></ul>
<ul> <li><b>Xh/Yh:</b> calculates a corrected horizontal X magnetic component & Y magnetic component</li> </ul>
<ul> <li><b>yaw:</b> calculates yaw from the corrected magnetometer measurements, representing compass-like rotation around the vertical z-axis</li> </ul>
<ul> <li><b>R:</b> the matrices are combined into one total rotation matrix</li></ul>
<ul> <li><b>x_rot/y_rot/z_rot:</b> each original axis vector is multiplied by the total rotation matrix</li> </ul>

    
<br>
<ul> <li><b>fig.clear():</b> removes everything drawn during the previous loop, without this line the graph would keep adding arrows on top of the old arrows</li> </ul>
<ul> <li><b>ax_3d:</b> creates a new 3D graph inside the figure</li> </ul>
<ul> <li><b>ax_3d.quiver:</b> draws the rotated arrows</li> </ul>
     <ul><ul><ul> <li>the first 3 values are the arrow's starting position, next three are its direction, length is by units, normalize=True maintains the arrow at a consistent length, linewidth makes the arrow thicker</li></ul></ul></ul>
<ul> <li><b>ax_3d.set_xlim/ylim/zlim:</b> these lines keep the graph's dimensions fixed, or else the animation could appear unstable with automatic resizing</li> </ul>
<ul> <li><b>ax_3d.set_title:</b> displays the current roll, pitch, and yaw</li> </ul>
    <ul><ul><ul> <li>.1f displays one digit after decimal point</li></ul></ul></ul>
<ul> <li><b>ax_3d.view_init:</b> controls where the viewer appears to be looking from</li> </ul>
    <ul><ul><ul> <li>elev & azim are in degrees, elev is with respect to horizontal plane & azim is with respect to around the graph</li></ul></ul></ul>
<ul> <li><b>clear_output/display:</b> clear removes the previously displayed graph, display displays the newly updated graph</li> </ul>
    <ul><ul><ul> <li>because these commands run repeatedly, the arrows appear to move in real time</li></ul></ul></ul>

<br>
<ul> <li><b>elapsed_time:</b> calculates how long the current loop took</li> </ul>
<ul> <li><b>sleep_time:</b> calculates how long the program should pause</li> </ul>
<ul> <li><b>time.sleep:</b> pauses the program for the remaining time, helping the program maintain an interval</li> </ul>
<ul> <li><b>except KeyboardInterrupt:</b> instead of showing a large error message, the program catches the interruption and prints a clean stopping message</li></ul>


</div>

In [ ]:
try:
    while True:
        start_time = time.time()


        # 5. get the newest 9-axis sensor data
        imu.fetch_motion9()

        # accelerometer data
        ax, ay, az = imu.accel_x, imu.accel_y, imu.accel_z

        # gyroscope data
        gx, gy, gz = imu.gyro_x, imu.gyro_y, imu.gyro_z

        # magnetometer data
        mx, my, mz = (
            imu.magneto_x,
            imu.magneto_y,
            imu.magneto_z
        )


        # 6. calculate roll and pitch using a complementary filter
        if az == 0:
            az = 0.000001

        roll_acc = math.atan2(ay, az) * 57.29578

        pitch_acc = math.atan2(
            -ax,
            math.sqrt(ay * ay + az * az)
        ) * 57.29578


        roll_filtered = (
            alpha * (roll_filtered + gx * dt)
            + (1.0 - alpha) * roll_acc
        )

        pitch_filtered = (
            alpha * (pitch_filtered + gy * dt)
            + (1.0 - alpha) * pitch_acc
        )


        # 7. calculate yaw using tilt-compensated magnetometer data
        r_rad = math.radians(roll_filtered)
        p_rad = math.radians(pitch_filtered)


        Xh = (
            mx * math.cos(p_rad)
            + my * math.sin(r_rad) * math.sin(p_rad)
            + mz * math.cos(r_rad) * math.sin(p_rad)
        )

        Yh = (
            my * math.cos(r_rad)
            - mz * math.sin(r_rad)
        )


        yaw = math.atan2(-Yh, Xh) * 57.29578

        # convert negative yaw angles into the range 0 to 360 degrees
        if yaw < 0:
            yaw += 360.0

        y_rad = math.radians(yaw)


        # 8. calculate the 3D rotation matrix
        # rotation order: yaw -> pitch -> roll

        # rotation around the X-axis
        R_x = np.array([
            [1, 0, 0],
            [0, math.cos(r_rad), -math.sin(r_rad)],
            [0, math.sin(r_rad), math.cos(r_rad)]
        ])


        # rotation around the Y-axis
        R_y = np.array([
            [math.cos(p_rad), 0, math.sin(p_rad)],
            [0, 1, 0],
            [-math.sin(p_rad), 0, math.cos(p_rad)]
        ])


        # rotation around the Z-axis
        R_z = np.array([
            [math.cos(y_rad), -math.sin(y_rad), 0],
            [math.sin(y_rad), math.cos(y_rad), 0],
            [0, 0, 1]
        ])


        # total rotation matrix: R = Rz × Ry × Rx
        R = np.dot(
            R_z,
            np.dot(R_y, R_x)
        )


        # calculate the rotated X, Y, and Z vectors
        x_rot = np.dot(R, x_axis_base)
        y_rot = np.dot(R, y_axis_base)
        z_rot = np.dot(R, z_axis_base)


        # 9. clear and redraw the 3D figure
        fig.clear()

        ax_3d = fig.add_subplot(
            111,
            projection='3d'
        )


        # draw the rotated X-axis arrow in red
        ax_3d.quiver(
            0, 0, 0,
            x_rot[0], x_rot[1], x_rot[2],
            color='red',
            length=1.0,
            normalize=True,
            linewidth=3,
            label='X-axis'
        )


        # draw the rotated Y-axis arrow in green
        ax_3d.quiver(
            0, 0, 0,
            y_rot[0], y_rot[1], y_rot[2],
            color='green',
            length=1.0,
            normalize=True,
            linewidth=3,
            label='Y-axis'
        )


        # draw the rotated Z-axis arrow in blue
        ax_3d.quiver(
            0, 0, 0,
            z_rot[0], z_rot[1], z_rot[2],
            color='blue',
            length=1.0,
            normalize=True,
            linewidth=3,
            label='Z-axis'
        )


        # set the fixed graph limits
        ax_3d.set_xlim([-1.2, 1.2])
        ax_3d.set_ylim([-1.2, 1.2])
        ax_3d.set_zlim([-1.2, 1.2])


        # label the graph axes
        ax_3d.set_xlabel('X')
        ax_3d.set_ylabel('Y')
        ax_3d.set_zlabel('Z')


        # show the current roll, pitch, and yaw values
        ax_3d.set_title(
            f"3D Attitude\n"
            f"R: {roll_filtered:.1f}°  "
            f"P: {pitch_filtered:.1f}°  "
            f"Y: {yaw:.1f}°"
        )


        ax_3d.legend(loc='upper left')


        # set the default viewing angle
        ax_3d.view_init(
            elev=20,
            azim=45
        )


        # display the updated figure in Jupyter
        clear_output(wait=True)
        display(fig)


        # 10. control the timing of each loop
        elapsed_time = time.time() - start_time

        sleep_time = max(
            0,
            dt - elapsed_time
        )

        time.sleep(sleep_time)


except KeyboardInterrupt:
    print("\nThe real-time 3D simulation has stopped.")

<div style="background-color: #fee4ec; padding: 12px; border-radius: 10px; font-size: 20px; font-family: Times;">
<h3 style="font-size: 24px; font-weight: bold;">How is the IMU Able to Function the Way it Does</h3>

<div style="background-color: #fee4ec; padding: 12px; border-radius: 10px; font-size: 16px; font-family: Times;">

<ul> <li><b>IMU works by combinign three tiny sensors inside one module:</b></li> </ul>
    <ul><ul><ul> <li>a 3-axis accelerometer</li></ul></ul></ul>
    <ul><ul><ul> <li>a 3-axis gyroscope</li></ul></ul></ul>
    <ul><ul><ul> <li>a 3-axis magnetometer</li></ul></ul></ul>

<br>
<ul> <li><b>accelerometer:</b> detects gravity & staright-line motion</li> </ul>
    <ul><ul><ul> <li>inside the sensor is an extremely small movable mass made using MEMS (microelectromechanical systems) technology</li></ul></ul></ul>
    <ul><ul><ul> <li>when the sensor moves or tilts, the tiny mass shifts slightly, the chip detects this movement as a change in electrical capacitance and converts it into acceleration data</li></ul></ul></ul>
    <ul><ul><ul> <li>when the IMU is sitting still, the accelerometer still detects gravity</li></ul></ul></ul>
    <ul><ul><ul> <li>accelerometer is good at providing a long-term reference, but it can become noisy when the sensor shakes or moves quickly</li></ul></ul></ul>

    
    
<br>
<ul> <li><b>gyroscope:</b> detects rotation speed</li> </ul>
    <ul><ul><ul> <li>inside a MEMS gyroscope is a microscopic structure that vibrates continuously</li></ul></ul></ul>
    <ul><ul><ul> <li>when the sensor rotates, the moving structure experiences the Coriolis effect, causing it to shift in another direction, the chip measures this tiny shift and calculates angular velocity</li></ul></ul></ul>
    <ul><ul><ul> <li>gyroscope responds quickly and smoothly, but small measurement errors accumulate over time, this is called gyroscope drift</li></ul></ul></ul>
    

<br>
<ul> <li><b>magnetometer:</b> detects magnetic direction</li> </ul>
    <ul><ul><ul> <li>works like a small electronic compass</li></ul></ul></ul>
    <ul><ul><ul> <li>depending on the sensor design, it detects magnetism using a magnetoresistive or similar microscopic sensing element, the electrical properties of the element change depending on the direction and strength of the magnetic field</li></ul></ul></ul>
    <ul><ul><ul> <li>because Earth has a magnetic field, the magnetometer can provide a reference direction for calculating yaw</li></ul></ul></ul>
    <ul><ul><ul> <li>nearby objects can interfere with yaw such as magnets, motors, speakers, metal objects, electrical wires, the PYNQ board itself, which can make yaw unstable or inaccurate unless the magnetometer is calibrated</li></ul></ul></ul>


<br>
<ul> <li><b>converting physical movement into numbers:</b></li> </ul>
    <l>each internal sensor first produces a very small electrical signal, then the IMU’s internal electronics:</l>
    <ul><ul><ul> <li>measure the sensor signal</li></ul></ul></ul>
    <ul><ul><ul> <li>convert the analog signal into digital data</li></ul></ul></ul>
    <ul><ul><ul> <li>store the data in internal registers</li></ul></ul></ul>
    <ul><ul><ul> <li>send the values to the PYNQ-Z2</li></ul></ul></ul>
<br><ul><ul><ul> <li>the PYNQ board communicates with the sensor using I2C, commonly using SDA (serial data) & SCL (serial clock)</li></ul></ul></ul>


<br>
<ul> <li><b>imperfections:</b></li> </ul>
    <ul><ul><ul> <li>the accelerometer can estimate roll and pitch, but it cannot reliably determine yaw because gravity always points vertically and does not provide a horizontal compass direction</li></ul></ul></ul>
    <ul><ul><ul> <li>the gyroscope can track all three rotations, but its errors build up</li></ul></ul></ul>
    <ul><ul><ul> <li>the magnetometer helps correct yaw by providing a magnetic direction reference</li></ul></ul></ul>
    
<br>
<ul> <li><b>sensor fusion:</b></li> </ul>
    <l>complementary filter: combining multiple sensors to produce a better orientation estimate</l>
    <ul><ul><ul> <li>the gyroscope handles fast movement, while the accelerometer slowly corrects gyroscope drift</li></ul></ul></ul>
    <ul><ul><ul> <li>yaw is calculated using the magnetometer after correcting for the sensor’s roll and pitch</li></ul></ul></ul>
    
<br>
<ul> <li><b>3D orientation:</b></li> </ul>
    <l>after finding roll, pitch, and yaw, the program creates three rotation matrices:</l>
    <ul><ul><ul> <li>x-axis for roll</li></ul></ul></ul>
    <ul><ul><ul> <li>y-axis for pitch</li></ul></ul></ul>
    <ul><ul><ul> <li>z-axis for yaw</li></ul></ul></ul>
    <ul><ul><ul> <li>they are then combined into one matrix</li></ul></ul></ul>
    
</div>